In [12]:
#setup-imports-paths-and-piqs-detection
from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import qutip as qt
import qutip.piqs as piqs

from IPython.display import display, Math

#project-root-path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

#figures-folder-path
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

#outputs-folder-path
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Figures folder:", FIGURES_DIR)
print("Outputs folder:", OUTPUTS_DIR)
print("QuTiP version:", qt.__version__)
print("qutip.piqs location:", piqs.__file__)
print("PIQS module imported.")

Project root: c:\Users\ivraa\Documents\GitHub\optics-master-equation-project
Figures folder: c:\Users\ivraa\Documents\GitHub\optics-master-equation-project\figures
Outputs folder: c:\Users\ivraa\Documents\GitHub\optics-master-equation-project\outputs
QuTiP version: 5.2.3
qutip.piqs location: C:\Users\ivraa\miniforge31\envs\optics-me\Lib\site-packages\qutip\piqs\piqs.py
PIQS module imported.


In [13]:
#piqs-fallback-imports
PIQS_IMPORT_STYLE = None

try:
    from qutip.piqs import Dicke, jspin, excited
    PIQS_IMPORT_STYLE = "from qutip.piqs import Dicke, jspin, excited"
    print("Import style 1 worked.")
except Exception as error1:
    print("Import style 1 failed:", error1)

if PIQS_IMPORT_STYLE is None:
    try:
        from qutip.piqs.piqs import Dicke, jspin, excited
        PIQS_IMPORT_STYLE = "from qutip.piqs.piqs import Dicke, jspin, excited"
        print("Import style 2 worked.")
    except Exception as error2:
        print("Import style 2 failed:", error2)

if PIQS_IMPORT_STYLE is None:
    try:
        from piqs import Dicke, jspin, excited
        PIQS_IMPORT_STYLE = "from piqs import Dicke, jspin, excited"
        print("Import style 3 worked.")
    except Exception as error3:
        print("Import style 3 failed:", error3)

if PIQS_IMPORT_STYLE is None:
    raise ImportError("No PIQS import style worked. We need to inspect dir(qutip.piqs).")
else:
    print("Using:", PIQS_IMPORT_STYLE)

Import style 1 failed: cannot import name 'Dicke' from 'qutip.piqs' (C:\Users\ivraa\miniforge31\envs\optics-me\Lib\site-packages\qutip\piqs\__init__.py)
Import style 2 worked.
Using: from qutip.piqs.piqs import Dicke, jspin, excited


In [14]:
#physical-setup-piqs-gamma0-effects

#gammac-collective-cavity-decay-rate-parameter
gammac = 1.0

#gamma0-over-gammac-values-parameter
gamma0_ratios = [0.0, 0.05, 0.1, 0.5, 1.0, 2.0]

#dimensionless-time-s-gammac-t
s_min = 0.0
s_max = 3.0
num_points = 500
s_grid = np.linspace(s_min, s_max, num_points)

#physical-time-grid-t=s/gammac
t_grid = s_grid / gammac

print("gammac =", gammac)
print("gamma0/gammac ratios =", gamma0_ratios)
print("time points =", len(t_grid))
print("Using dimensionless time s = Gamma_c t")

gammac = 1.0
gamma0/gammac ratios = [0.0, 0.05, 0.1, 0.5, 1.0, 2.0]
time points = 500
Using dimensionless time s = Gamma_c t


In [15]:
#latex-piqs-mapping
display(Math(r"L_c = \sqrt{\Gamma_c}J_-"))
display(Math(r"J_-=\sum_i \sigma_-^i"))
display(Math(r"L_i=\sqrt{\Gamma_0}\sigma_-^i"))
display(Math(r"\mathrm{PIQS:}\quad \mathrm{collective\_emission}=\Gamma_c,\quad \mathrm{emission}=\Gamma_0"))
display(Math(r"P_e(t)=\langle J_z\rangle+\frac{N}{2}\quad \mathrm{or}\quad P_e(t)=\frac{N}{2}-\langle J_z\rangle"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [16]:
#solve-piqs-gamma0-effect-function
def solve_piqs_gamma0_effect(num_atoms, gamma0, gammac, t_grid, jz_sign="plus"):
    """
    #piqs-master-equation-solver
    Solves the permutationally invariant master equation for N identical atoms.

    #collective-emission
    collective_emission = gammac corresponds to the cavity channel.

    #local-emission
    emission = gamma0 corresponds to independent free-space decay.

    #observable
    P_e(t) is inferred from <J_z>.
    We test whether the correct convention is:
    P_e = <J_z> + N/2
    or
    P_e = N/2 - <J_z>
    """
    
    #dicke-ensemble-calc
    ensemble = Dicke(
        N=num_atoms,
        emission=gamma0,
        collective_emission=gammac,
    )
    
    #liouvillian-calc
    liouvillian = ensemble.liouvillian()
    
    #initial-state-all-excited-dicke-calc
    rho0 = excited(num_atoms)
    
    #collective-spin-z-operator-calc
    jz = jspin(num_atoms, "z")
    
    #solve-master-equation-calc
    result = qt.mesolve(
        liouvillian,
        rho0,
        t_grid,
        [],
        e_ops=[jz],
    )
    
    #jz-expectation-calc
    jz_expectation = np.real(result.expect[0])
    
    if jz_sign == "plus":
        #excited-population-convention-plus-calc
        total_excited_population = jz_expectation + num_atoms / 2.0
    
    elif jz_sign == "minus":
        #excited-population-convention-minus-calc
        total_excited_population = num_atoms / 2.0 - jz_expectation
    
    else:
        raise ValueError("jz_sign must be 'plus' or 'minus'")
    
    return total_excited_population

In [17]:
#piqs-jz-convention-test
test_N = 6
test_ratio = 0.5
test_gamma0 = test_ratio * gammac

start_time = time.time()

population_plus = solve_piqs_gamma0_effect(
    num_atoms=test_N,
    gamma0=test_gamma0,
    gammac=gammac,
    t_grid=t_grid,
    jz_sign="plus",
)

population_minus = solve_piqs_gamma0_effect(
    num_atoms=test_N,
    gamma0=test_gamma0,
    gammac=gammac,
    t_grid=t_grid,
    jz_sign="minus",
)

elapsed_time = time.time() - start_time

print("Testing PIQS Jz convention for N =", test_N)
print("Expected initial excited population:", test_N)
print("Runtime seconds:", elapsed_time)
print()
print("plus convention initial:", population_plus[0])
print("plus convention final:", population_plus[-1])
print()
print("minus convention initial:", population_minus[0])
print("minus convention final:", population_minus[-1])

Testing PIQS Jz convention for N = 6
Expected initial excited population: 6
Runtime seconds: 0.0676276683807373

plus convention initial: 6.0
plus convention final: 0.17086822695886905

minus convention initial: 0.0
minus convention final: 5.829131773041131


In [18]:
#choose-correct-jz-convention
JZ_SIGN = "plus"   #change to "minus" if Cell 6 shows minus is correct

print("Using JZ_SIGN =", JZ_SIGN)

Using JZ_SIGN = plus


In [19]:
#piqs-quick-test-N6
test_N = 6
test_ratio = 0.5
test_gamma0 = test_ratio * gammac

start_time = time.time()

population_test = solve_piqs_gamma0_effect(
    num_atoms=test_N,
    gamma0=test_gamma0,
    gammac=gammac,
    t_grid=t_grid,
    jz_sign=JZ_SIGN,
)

elapsed_time = time.time() - start_time

print("N =", test_N)
print("gamma0/gammac =", test_ratio)
print("runtime seconds =", elapsed_time)
print("initial population =", population_test[0])
print("final population =", population_test[-1])

N = 6
gamma0/gammac = 0.5
runtime seconds = 0.02749800682067871
initial population = 6.0
final population = 0.17086822695886905


In [20]:
#run-piqs-gamma0-sweep-moderate-N
atom_numbers_piqs = [10, 12, 16]

piqs_populations = {}
piqs_runtime_rows = []

for num_atoms in atom_numbers_piqs:
    
    piqs_populations[num_atoms] = {}
    
    for ratio in gamma0_ratios:
        
        gamma0_current = ratio * gammac
        
        print("\n" + "=" * 60)
        print(f"Solving PIQS: N={num_atoms}, gamma0/gammac={ratio}")
        
        start_time = time.time()
        
        population = solve_piqs_gamma0_effect(
            num_atoms=num_atoms,
            gamma0=gamma0_current,
            gammac=gammac,
            t_grid=t_grid,
            jz_sign=JZ_SIGN,
        )
        
        elapsed_time = time.time() - start_time
        
        piqs_populations[num_atoms][ratio] = population
        
        piqs_runtime_rows.append({
            "N": num_atoms,
            "gamma0/gammac": ratio,
            "runtime seconds": elapsed_time,
            "initial population": population[0],
            "final population": population[-1],
        })
        
        np.save(
            OUTPUTS_DIR / f"piqs_population_N{num_atoms}_ratio_{ratio}.npy",
            population,
        )
        
        print(f"Finished in {elapsed_time:.3f} seconds")

piqs_runtime_table = pd.DataFrame(piqs_runtime_rows)

runtime_csv_path = OUTPUTS_DIR / "piqs_gamma0_effect_runtime_table_moderate_N.csv"
piqs_runtime_table.to_csv(runtime_csv_path, index=False)

print("Saved runtime table to:", runtime_csv_path)
piqs_runtime_table


Solving PIQS: N=10, gamma0/gammac=0.0
Finished in 0.046 seconds

Solving PIQS: N=10, gamma0/gammac=0.05
Finished in 0.040 seconds

Solving PIQS: N=10, gamma0/gammac=0.1
Finished in 0.036 seconds

Solving PIQS: N=10, gamma0/gammac=0.5
Finished in 0.028 seconds

Solving PIQS: N=10, gamma0/gammac=1.0
Finished in 0.061 seconds

Solving PIQS: N=10, gamma0/gammac=2.0
Finished in 0.048 seconds

Solving PIQS: N=12, gamma0/gammac=0.0
Finished in 0.060 seconds

Solving PIQS: N=12, gamma0/gammac=0.05
Finished in 0.049 seconds

Solving PIQS: N=12, gamma0/gammac=0.1
Finished in 0.077 seconds

Solving PIQS: N=12, gamma0/gammac=0.5
Finished in 0.065 seconds

Solving PIQS: N=12, gamma0/gammac=1.0
Finished in 0.069 seconds

Solving PIQS: N=12, gamma0/gammac=2.0
Finished in 0.063 seconds

Solving PIQS: N=16, gamma0/gammac=0.0
Finished in 0.141 seconds

Solving PIQS: N=16, gamma0/gammac=0.05
Finished in 0.144 seconds

Solving PIQS: N=16, gamma0/gammac=0.1
Finished in 0.141 seconds

Solving PIQS: N=16, g

,N,gamma0/gammac,runtime seconds,initial population,final population
0,10,0.00,0.046485,10.0,3.346408e-10
1,10,0.05,0.040210,10.0,8.419947e-02
2,10,0.10,0.036131,10.0,1.469747e-01
3,10,0.50,0.028072,10.0,2.398838e-01
4,10,1.00,0.061002,10.0,1.080272e-01
5,10,2.00,0.048455,10.0,9.561050e-03
6,12,0.00,0.060191,12.0,2.305267e-11
7,12,0.05,0.048519,12.0,9.170398e-02
8,12,0.10,0.076821,12.0,1.599216e-01
9,12,0.50,0.064979,12.0,2.625890e-01
